# Agent

エージェントは、ユーザー・LLM・外部ツールのやり取りを仲介して一連のワークフローを制御する、LangChainの中核となるコンポーネントです。

<img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/core_agent_loop.svg?w=1650&fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=cb329408a32e8d35223b7a33bc407785" width=30%>

エージェントはユーザーとLLMの会話を以下のような単位で管理します

- thread: 一連の会話の流れ（ChatGPTやClaude Codeの左側にリスト表示されているスレッドとほぼ同概念）。thread内の情報を保持するのがcheckpointer
- message: 一度の会話（ChatGPTやClaude Codeの一度の質問と回答のペアが相当）
- （長期メモリ: ユーザID等に基づきthread間で引き継がれる情報。主にLangGraphが担当）

エージェントは、LLM本体であるmodelと、その動作を制御するharness群からなります。

- model: LLM本体
- harness: LLMの役割や入出力や状態を制御する。以下構成要素からなる
    - system plompt: modelがどのような**役割**で回答するかを指定
    - tools: modelが**参照できる関数やLangChain Tool**を指定（エージェント作成時に指定するため、**message/threadをまたいで適用**される）
    - context: messageにおいて**modelが参照できる情報**を指定（データそのものに加え、DBセッションやAPIなどの参照先も渡せる。**単一のmessageに限られる**）
    - memory: message間で**履歴情報を引き継ぐ**仕組み。thread内の履歴を引き継ぐcheckpointerが代表的（メモリ処理自体は主にLangGraphが担当）
    - skills
    - subagents

<img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/agent_model_harness.svg?w=1650&fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=206e8f37e9b0ddf8dde21270c1e1d333" width=40%>

エージェントにメッセージを送って返答を得る方法は、主に以下の2種類があります。

- invoke: 回答の生成が全て終わってから一括返信
- streaming: 回答の途中経過を逐次返信

## harnesses

各種ハーネスの概要

### Tools

modelが常に参照できる関数やLangChain Toolsを指定します（Toolsについては別ノートブックで詳細解説）。`langchain.agents.create_agent`関数の`tools`引数に渡すことでToolsが参照されるようになります。
なお、LocalLLM（`langchain_huggingface.ChatHuggingFace`）では`tools`引数に指定するだけではうまく動かないことが多いです（VLLM等を使用する必要）

In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)

### System prompt

LLMが回答を返す際の役割や前提条件に相当するものを指定します。基本的にはLLMの`<system>`プロンプトに渡される内容に相当します。

In [ ]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    system_prompt="You are a helpful assistant who prefers concise answers",
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the currency of Japan?"}]}
)
print(result["messages"][-1].content_blocks)

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    system_prompt="You are a cynical assistant",
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the currency of Japan?"}]}
)
print(result["messages"][-1].content_blocks)

### Structured output

返答のフォーマット（フィールドや型）をPydanticで定義して`langchain.agents.create_agent`関数の`response_format`引数に渡すことで、指定フォーマットに従った返答が返るようにできます。

In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent

class Answer(BaseModel):
    summary: str
    confidence: float

agent = create_agent(model="google_genai:gemini-2.5-flash-lite", response_format=Answer)
result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
result["structured_response"]

### checkpointer (memory)

thread内の複数のmessage間で履歴を保持するために利用されるのが、checkpointerです。checkpointerを使用する場合、以下の実装が必要となります。

- `langchain.agents.create_agent`関数の`checkpointer`引数にcheckpointerを渡す
- メッセージ送信時（`invoke`または`astream_events`メソッド）に、`config`引数に`thread_id`を渡す

checkpointerはスクリプトが動作するサーバのメモリ内で管理するシンプルな`langgraph.checkpoint.memory.InMemorySaver`がよく利用されますが、PostgreSQLに保持する`langgraph.checkpoint.postgres.PostgresSaver`や`langgraph.checkpoint.sqlite.SqliteSaver`も存在します（これらは厳密にはLangGraphの機能です）。

具体的な動作は後述します。

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[],
    checkpointer=InMemorySaver(),
)

## Agentへのメッセージ送受信

前述のように、Agentへのメッセージ送受信には返答をまとめて受信するinvokeと、逐次的に受信するstreamerが存在します。

### invoke

Agentにメッセージを送り、回答生成を待って全文を受け取るには`invoke`メソッドを使用します。
thread内で履歴を引き継ぎたい場合、`langchain.agents.create_agent`関数の`checkpointer`引数に`langgraph.checkpoint.memory.InMemorySaver`を、`invoke`メソッドの`config`引数に`thread_id`を渡す必要があります。

In [ ]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]},
    config=config,
)

# A follow-up turn on the same conversation: reuse the same thread_id to keep history
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What about tomorrow?"}]},
    config=config,
)
print(result)

戻り値は以下のように、今までの質問と返答がリスト形式で返されます。

```python
{'messages':[
    HumanMessage(content="What's the weather in San Francisco?",...)
    AIMessage(content='I can help you with that! Please tell me what you mean by "San Francisco."...', response_metadata={...},...)
    HumanMessage(content='What about tomorrow?',...)
    AIMessage(content='"Okay, I can tell you about the weather in San Francisco, California...', response_metadata={...},...)
]}
```

#### thread_idとcheckpointer

上の例では同一のthread_idを渡していたため、全messageが同一のthreadとみなされていました。一方で、異なるthread_idを`invoke`メソッドに渡すと別threadとみなされます。

以下の例では、thread_idが同じであれば会話の内容が引き継がれ、別のthread_idを指定すると会話内容が引き継がれないことが分かります。

In [ ]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[],
    checkpointer=InMemorySaver(),
)

config1 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]},
    config=config1,
)

# A follow-up turn on the same conversation: reuse the same thread_id to keep history
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What about tomorrow?"}]},
    config=config1,
)
print(result)

# New thread: use a different thread_id to start a new conversation
config2 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What about the day after tomorrow?"}]},
    config=config2,
)
print(result)

#### Streaming

回答の途中経過を逐次得るには、`stream_events`メソッドを使用します。`config`引数に`thread_id`を渡すのを忘れないようにしてください。またバージョンにより挙動が変わるので、`version`引数に"v3"を指定してください。

[公式のAgentの項にあるコード](https://docs.langchain.com/oss/python/langchain/agents#streaming)は動作しなかったので、[Event streamingの項にあるコード](https://docs.langchain.com/oss/python/langchain/event-streaming)を基に以下の実装で、行ごとに回答を逐次printできます

In [ ]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]},
    version="v3",
    config=config,
)
for message in stream.messages:
    for delta in message.text:
        print(delta, end="", flush=True)

チャットボットを作る場合、このストリーム返答をServer-Sent Events (SSE)でフロントエンドに返すと良いでしょう。FastAPIを使用する場合、以下のように実装できます（今回の`for message in stream.messages`では滑らかな文字単位ではなくチャンク単位での送信となるためご注意ください）。

```python
import json

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

app = FastAPI()

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[],
    checkpointer=InMemorySaver(),
)

class ChatRequest(BaseModel):
    message: str
    thread_id: str | None = None

def sse(data: dict, event: str = "message") -> str:
    return f"event: {event}\ndata: {json.dumps(data, ensure_ascii=False)}\n\n"

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    thread_id = req.thread_id or str(uuid7())
    config = {"configurable": {"thread_id": thread_id}}
    # Text stream generator
    async def text_generator():
        try:
            stream = agent.astream_events(
                {"messages": [{"role": "user", "content": req.message}]},
                version="v3",
                config=config,
            )
            yield sse({"thread_id": thread_id}, event="metadata")
            # Stream loop
            async for message in stream.messages:
                for delta in message.text:
                    yield sse({"delta": delta}, event="token")
            # Finish
            yield sse({"done": True}, event="done")

        # Error handling
        except Exception as e:
            yield sse({"error": str(e)}, event="error")

    # Return to the frontend
    return StreamingResponse(
        text_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",  # nginx対策
        },
    )
```

## ハーネス構成

上で紹介したtools, structed output, checkpointer以外にも、LangChainのハーネスには様々な便利機能があります。

<img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/agent_harness_capabilities.svg?w=1650&fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=909587f6275319afc54cb6d2ed7a363f" width=60%>

### 実行環境

LLMは高い言語理解とテキスト生成能力を持ちますが、正確な計算やデータ処理は苦手です。この能力をカバーするためには、AIがPython等のコードを生成して計算やデータ処理を行わせることが有効です。このようなコード実行のために、LangChainでは以下の機能が準備されています。

- Interpreter: エージェントにコードを生成・実行させるための能力を持たせる
- Sandboxes: 生成したPythonコードを安全に実行させるための閉じた実行環境
- Filesystem: ユーザがファイルをアップロードしてLLMが読み込めるようにしたり、LLMがデータの処理結果を出力するフォルダ

### Context management

checkpointerでthread_idを指定すると、thread内の過去のメッセージ履歴を考慮した回答できるようになります。
この**履歴情報をどのように管理してLLMに入力するかをマネジメントする**ために、以下の機能が準備されています

- MemoryMiddleware: 過去の会話履歴を保存してLLMの入力プロンプトに差し込む（デフォルト状態の`create_agent`とcheckpointerでも履歴引継機能はあるが、明示的に指定することでトークン・コスト制限やSummarizationとの連携を柔軟に設定できる）
- SummarizationMiddleware: 会話が長くなりすぎたときに、LLM自身に要約させて古いメッセージと差し替える
- Skills: 外部知識を記述した文章を、LLMが自動参照できるようにする。Claude Codeのようにmarkdown形式で読み込ませることも可能（Skills用のMiddlewareがあるわけではなく、汎用のAgentMiddlewareを継承して作成する。「機能」というより「設計パターン」に近い）

### Planning and delegation

LLMのコンテキスト長には限界があるため、1つのLLMだけでthreadを全て処理しようとするとパンクします。
これを防ぐためには複数のLLMに分業させる仕組みが有効で、LangChainではメインのLLM（Agent）以外にサブのLLMである**subagents**を利用することで、Agentのコンテキストを簡潔に保つことを目指します。

### Fault tolerance



### subagents

LangChainではメインのLLM（Agent）に加えてサブのLLMであるsubagentsを利用することで、分業によりコンテキスト長を有効活用できるようになります。
このとき、メインのAgentはプロンプトから想定されるタスクをsubagentsに仕分ける作業に専念し、subagentsはこの割り振られたタスクをこなしてメインAgentに返します。

なお、このような複数のLLMの分業はLangGraphのNodeコンセプトとよく似ていますが、両者は以下のような差があります。

|比較項目|LangChainの subagents（旧・伝統的アプローチ）|LangGraphの Nodeによるマルチエージェント（モダン）|
|---|---|---|
|構造の表現|入れ子構造（親子関係） のチェイン。|グラフ構造（ネットワーク型） のノード遷移。|
|状態（State）の管理|独立（親は子の途中経過を知らず、最終結果だけを受け取る）。|共有の State を全員で読み書き、または明示的にパースして遷移。|
|会話のやり取り|一方通行（親 ➔ 子 ➔ 親）。子がさらに別の担当へパスするのは困難。|自由なキャッチボール（A ➔ B ➔ C ➔ Aなど、動的なループが可能）。|
|適したタスク|1回の実行（invoke）で完結するタスク向き。|ユーザーの介入（Human-in-the-loop）や一時停止、承認が必要なシステム向き。|

ざっくり言うとLangChainのsubagentsはAgent明確な上下関係があり、LangGraphのNodeは並列で分散された関係にあります。

In [ ]:
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware
from deepagents.middleware.subagents import SubAgentMiddleware
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Search for a query and return a short summary."""
    return f"Search results for: {query}"


backend = StateBackend()

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[search],
    middleware=[
        FilesystemMiddleware(backend=backend),
        TodoListMiddleware(),
        SubAgentMiddleware(
            backend=backend,
            subagents=[
                {
                    "name": "researcher",
                    "description": "Searches and returns a structured summary.",
                    "system_prompt": "Use the search tool to research the question and summarize key points.",
                    "tools": [search],
                    "model": "anthropic:claude-sonnet-4-6",
                    "middleware": [],
                }
            ],
        ),
    ],
)


### Fault tolerance

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Search for a query and return a short summary."""
    return f"Search results for: {query}"


agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[search],
    middleware=[
        ModelRetryMiddleware(max_retries=3),
        ToolRetryMiddleware(max_retries=2),
    ],
)

## Skills

外部知識を記述した文章を、それに近いLLMが自動参照できるようにすることで、。Claude Codeのようにmarkdown形式で読み込ませることも可能です。
注意点として、skills用のMiddlewareがあるわけではなく、汎用のAgentMiddlewareとtoolを組み合わせてskillsの概念を実装します。よってskillsはLangChain公式の「機能」というより「設計パターン」に近いとみなせるでしょう。

一例として[公式チュートリアルに従い](https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant#google-gemini)、「SQL文を書いてください」という指示を受け取ったエージェントが、スキーマを記述したskillに基づきSQL文を生成するプログラムを作ってみます。

まず以下のように、スキーマを記述スキルを定義します。
動作時には`name`と`description`が`<system>`プロンプトに組み込まれるため、質問文に`name`＆`description`に関する内容が含まれていれば、LLMが`content`の内容を参照するはずです。

In [ ]:
from typing import TypedDict

class Skill(TypedDict):
    """A skill that can be progressively disclosed to the agent."""
    name: str  # Unique identifier for the skill
    description: str  # 1-2 sentence description to show in system prompt
    content: str  # Full skill content with detailed instructions

SKILLS: list[Skill] = [
    {
        "name": "sales_analytics",
        "description": "Database schema and business logic for sales data analysis including customers, orders, and revenue.",
        "content": """# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active customers**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**Revenue calculation**: Only count orders with status = 'completed'. Use total_amount from orders table, which already accounts for discounts.

**Customer lifetime value (CLV)**: Sum of all completed order amounts for a customer.

**High-value orders**: Orders with total_amount > 1000

## Example Query

-- Get top 10 customers by revenue in the last quarter
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10;
""",
    },
    {
        "name": "inventory_management",
        "description": "Database schema and business logic for inventory tracking including products, warehouses, and stock levels.",
        "content": """# Inventory Management Schema

## Tables

### products
- product_id (PRIMARY KEY)
- product_name
- sku
- category
- unit_cost
- reorder_point (minimum stock level before reordering)
- discontinued (boolean)

### warehouses
- warehouse_id (PRIMARY KEY)
- warehouse_name
- location
- capacity

### inventory
- inventory_id (PRIMARY KEY)
- product_id (FOREIGN KEY -> products)
- warehouse_id (FOREIGN KEY -> warehouses)
- quantity_on_hand
- last_updated

### stock_movements
- movement_id (PRIMARY KEY)
- product_id (FOREIGN KEY -> products)
- warehouse_id (FOREIGN KEY -> warehouses)
- movement_type (inbound/outbound/transfer/adjustment)
- quantity (positive for inbound, negative for outbound)
- movement_date
- reference_number

## Business Logic

**Available stock**: quantity_on_hand from inventory table where quantity_on_hand > 0

**Products needing reorder**: Products where total quantity_on_hand across all warehouses is less than or equal to the product's reorder_point

**Active products only**: Exclude products where discontinued = true unless specifically analyzing discontinued items

**Stock valuation**: quantity_on_hand * unit_cost for each product

## Example Query

-- Find products below reorder point across all warehouses
SELECT
    p.product_id,
    p.product_name,
    p.reorder_point,
    SUM(i.quantity_on_hand) as total_stock,
    p.unit_cost,
    (p.reorder_point - SUM(i.quantity_on_hand)) as units_to_reorder
FROM products p
JOIN inventory i ON p.product_id = i.product_id
WHERE p.discontinued = false
GROUP BY p.product_id, p.product_name, p.reorder_point, p.unit_cost
HAVING SUM(i.quantity_on_hand) <= p.reorder_point
ORDER BY units_to_reorder DESC;
""",
    },
]

skillを読み込むためのtoolを作成します。

In [ ]:
from langchain.tools import tool

@tool
def load_skill(skill_name: str) -> str:
    """Load the full content of a skill into the agent's context.

    Use this when you need detailed information about how to handle a specific
    type of request. This will provide you with comprehensive instructions,
    policies, and guidelines for the skill area.

    Args:
        skill_name: The name of the skill to load (e.g., "expense_reporting", "travel_booking")
    """
    # Find and return the requested skill
    for skill in SKILLS:
        if skill["name"] == skill_name:
            return f"Loaded skill: {skill_name}\n\n{skill['content']}"

    # Skill not found
    available = ", ".join(s["name"] for s in SKILLS)
    return f"Skill '{skill_name}' not found. Available skills: {available}"

skill読込用のMilddleware（langchain.agents.middleware）を作成し、上で作成した`load_skill`ツールを入れます。
また`__init__`メソッドでSKILLSの`name`と`description`を実際のシステムプロンプトに入れる形に整形します。
さらに、`wrap_model_call`の部分でシステムプロンプトに上書きする処理を実装します（`skills_addendum`で、「skillsを必要に応じて読み込んでください」という旨のメッセージもシステムプロンプトに加えています）。

In [ ]:
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage
from typing import Callable

class SkillMiddleware(AgentMiddleware):
    """Middleware that injects skill descriptions into the system prompt."""

    # Register the load_skill tool as a class variable
    tools = [load_skill]

    def __init__(self):
        """Initialize and generate the skills prompt from SKILLS."""
        # Build skills prompt from the SKILLS list
        skills_list = []
        for skill in SKILLS:
            skills_list.append(
                f"- **{skill['name']}**: {skill['description']}"
            )
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """Sync: Inject skill descriptions into system prompt."""
        # Build the skills addendum
        skills_addendum = (
            f"\n\n## Available Skills\n\n{self.skills_prompt}\n\n"
            "Use the load_skill tool when you need detailed information "
            "about handling a specific type of request."
        )

        # Append to system message content blocks
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}
        ]
        new_system_message = SystemMessage(content=new_content)
        modified_request = request.override(system_message=new_system_message)
        return handler(modified_request)

作成したMilddlewareを`middleware`引数に設定したAgentを作成します。

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Create the agent with skill support
agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt=(
        "You are a SQL query assistant that helps users "
        "write queries against business databases."
    ),
    middleware=[SkillMiddleware()],
    checkpointer=InMemorySaver(),
)

「先月$1000以上の注文をした顧客を探すSQLを作成してください」というメッセージをエージェントに送ります。skills内のスキーマに基づき適切なSQL文が返ってくれば成功です。

In [ ]:
from langchain_core.utils.uuid import uuid7

# Configuration for this conversation thread
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

# Ask for a SQL query
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Write a SQL query to find all customers "
                    "who made orders over $1000 in the last month"
                ),
            }
        ]
    },
    config=config
)

# Print the conversation
for message in result["messages"]:
    if message.type == "ai":
        print("============================ AI Message ============================")
        if isinstance(message.content, list):
            text = "".join(
                block.get("text", "")
                for block in message.content
                if block.get("type") == "text"
            )
            if text:
                print(text)
        else:
            print(message.content)
    elif message.type == "human":
        print("============================ USER Message ============================")
        print(f"User: {message.content}")
    elif message.type == "tool":
        print("============================ TOOL Message ============================")
        print(f"Tool {message.name}: {message.content}")